In [2]:
import cv2 as cv, os

In [2]:
path = 'photos/image.png'
img = cv.imread(path)

width,height,channels = img.shape

totalPixels = width * height
dataType = img.dtype
disk_kb = os.path.getsize(path) / 1024
memory_kb = img.nbytes / 1024

In [3]:
print(f'width: {width}')
print(f'height: {height}')
print(f'channels: {channels}')
print(f'total pixels: {totalPixels}')
print(f'Data type : {dataType}')
print(f"On disk:   {disk_kb:.2f} KB")
print(f"In memory: {memory_kb:.2f} KB")

width: 954
height: 941
channels: 3
total pixels: 897714
Data type : uint8
On disk:   44.23 KB
In memory: 2630.02 KB


Load one image and display **4 versions** of it in 4 separate windows:
1. Original (BGR)
2. Grayscale
3. Flipped horizontally (look up `cv2.flip`)
4. Flipped vertically

In [8]:
img = cv.imread('photos/cat.jpg')


# 2. Define your explicit resolution (Width, Height)
width = 800
height = 600
dimensions = (width, height)

# 3. Resize the image
resized_img = cv.resize(img, dimensions, interpolation=cv.INTER_AREA)

cv.imshow('Cat',resized_img)
cv.waitKey(0)
cv.destroyAllWindows()


In [10]:
# 3. Save the resized image to a new file
# Syntax: cv2.imwrite(filename, img_array)
success = cv.imwrite('photos/cat_small.jpg', resized_img)

# 4. (Optional) Check if the save was successful
if success:
    print("Image saved successfully!")
else:
    print("Error: Could not save the image.")

Image saved successfully!


In [4]:
greyed_img = cv.imread('photos/cat_small.jpg',cv.IMREAD_GRAYSCALE)
cv.imshow('Greyed Cat',greyed_img)
cv.waitKey(0)
cv.destroyAllWindows()

In [5]:
img = cv.flip(greyed_img,0)
cv.imshow('Flipped Vertically', img)
cv.waitKey(0)
cv.destroyAllWindows()

In [7]:
img = cv.flip(greyed_img,1)
cv.imshow('Flipped Vertically', img)
cv.waitKey(0)
cv.destroyAllWindows()

Load a color image. Write code that turns a **50×50 pixel square** in each corner of the image to:
- Top-left → pure Red `(0, 0, 255)`
- Top-right → pure Green `(0, 255, 0)`
- Bottom-left → pure Blue `(255, 0, 0)`
- Bottom-right → pure White `(255, 255, 255)`

In [14]:
img = cv.imread('photos/cat_small.jpg')
height,width,channels = img.shape
# Top Left
img[0:50,0:50] = (0,0,255)
# Top Right
img[0:50,width-50:width] = (0,255,0)
# Bottom Left
img[height-50:height,0:50] = (255,0,0)
# Bottom Right
img[height-50:height,width-50:width] = (255,255,255)
success = cv.imwrite('photos/cat_pixel_setter.jpg',img)

if success:
    print("Image saved successfully!")
else:
    print('Error: Could not save the image.')

Image saved successfully!


### C4 — Channel Swapper
Load a color image. Using only **NumPy slicing** (no `cv2` color conversion functions), manually swap the Red and Blue channels so that the image looks like it was loaded in RGB order instead of BGR.

Display the original and the swapped version side by side and notice the color shift.

> **Constraint**: You cannot use `cv2.cvtColor`. Only use array indexing/copying.

In [20]:
import numpy as np
B = [[[1,2,3,4],
  [5,6,7,8],
  [9,10,11,12]],

 [[13,14,15,16],
  [17,18,19,20],
  [21,22,23,24]]]

B = np.array(B)
print(B)
print("=====================")
print(B[0:1])
print("=====================")
print(B[0:1,0:1])
print("=====================")
print(B[0:1,0:2])

[[[ 1  2  3  4]
  [ 5  6  7  8]
  [ 9 10 11 12]]

 [[13 14 15 16]
  [17 18 19 20]
  [21 22 23 24]]]
[[[ 1  2  3  4]
  [ 5  6  7  8]
  [ 9 10 11 12]]]
[[[1 2 3 4]]]
[[[1 2 3 4]
  [5 6 7 8]]]


In [41]:
imgBGR = cv.imread('photos/cat_small.jpg')
imgRGB = cv.imread('photos/cat_small.jpg')

imgRGB[:, :, [0, 2]] = imgRGB[:, :, [2, 0]]

success = cv.imwrite('photos/cat_channel_setter.jpg',imgRGB)

if success:
    print("Image saved successfully!")
else:
    print('Error: Could not save the image.')
# cv.imshow('BGR', imgBGR)
# cv.imshow('RGB', imgRGB)
# cv.waitKey(0)
# cv.destroyAllWindows()

Image saved successfully!


### C5 — Brightness Controller
Write a function:
```python
def adjust_brightness(image, factor):
    ...
```
- `factor > 1` → brighter
- `factor < 1` → darker
- `factor = 1` → no change

The function should **not** let any pixel value go above 255 or below 0 (look up `np.clip`).

Test it by displaying the same image at 5 different brightness levels.

> **Trap to avoid**: If you multiply a `uint8` array and the result exceeds 255, NumPy wraps it around (e.g., 260 becomes 4). You need to handle this carefully.

In [1]:
def adjust_brightness(image,factor):
    img_as_float = image.astype(float)
    img_as_float = img_as_float*factor
    img_as_int = np.clip(img_as_float.astype(int), a_min=0, a_max=255).astype(np.uint8)
    cv.imshow('Normal Image',image)
    cv.imshow('Adjusted brightness Image',img_as_int)
    cv.waitKey(0)
    cv.destroyAllWindows()

In [7]:
import cv2 as cv
import numpy as np
factor = float(input("Enter brightness factor between 0 and 1:"))
img = cv.imread('photos/cat_small.jpg')
adjust_brightness(img,factor)

In [44]:
import numpy as np

arr = np.array([1, 2, 5, 8, 10, -3])
result = np.clip(arr * 2, a_min=2, a_max=7)

print(result)
# Output: [2 2 5 7 7 2]

[2 4 7 7 7 2]


### C6 — Smart Cropper
Write a script that crops the **center** of any image to a fixed size of `(300, 300)`, regardless of the original image's dimensions.

Your script should work correctly whether the image is `(480, 640)` or `(1080, 1920)` — the crop should always be centered.

> **Think about it**: What is the center pixel? How far do you go in each direction from the center to get a 300×300 box?

In [20]:
img = cv.imread('photos/cat.jpg')
height,width,channels = img.shape 
center_y = height / 2
center_x = width / 2

start_y = center_y - 150
end_y = center_y + 150
start_x = center_x - 150
end_x = center_x + 150

cropped = img[int(start_y):int(end_y),int(start_x):int(end_x)]
width,height,channels = cropped.shape 
success = cv.imwrite('photos/cat_cropped_300x300.jpg',cropped)

if success:
    print("Image saved successfully!")
else:
    print('Error: Could not save the image.')

Image saved successfully!


### C7 — Blur Comparison Grid
Apply 5 different Gaussian blur levels to an image:
- Kernel sizes: `(3,3)`, `(11,11)`, `(21,21)`, `(41,41)`, `(71,71)`

Stack all 5 results into a **single horizontal strip** and save it as one image called `blur_comparison.jpg`.

> **Key concept**: Look up `np.hstack()`. But what's the constraint on arrays you can hstack? They need the same height. What if they already do?

In [27]:
img = cv.imread('photos/cat_small.jpg')
kernel_sizes = [(3,3),(11,11),(21,21),(41,41),(71,71)]
blurred_imgs_stack = []
for k in kernel_sizes:
    blurred_imgs_stack.append(cv.GaussianBlur(img,k,0))

success = cv.imwrite('photos/blur_comparison.jpg',np.hstack(blurred_imgs_stack))

if success:
    print("Image saved successfully!")
else:
    print('Error: Could not save the image.')
# cv.imshow('Blur Comparison',comparison_img_stack)


Image saved successfully!


## 🔴 Harder (You Need to Problem-Solve)

### C8 — Drawing with Data
Load an image. Find the pixel with the **highest brightness** in the entire image (convert to grayscale first, use `np.argmax`).

Draw:
- A red circle at that pixel's location
- A text label saying `"Brightest"` near the circle

> **Trap**: `np.argmax` on a 2D array returns a flat index. You need `np.unravel_index` to convert it to `(row, col)`. Remember that `cv2.circle` takes `(x, y)` = `(col, row)`.

In [22]:
import cv2 as cv,numpy as np 
img = cv.imread('photos/cat_small.jpg')
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)  

flat_idx = np.argmax(gray)
row, col = np.unravel_index(flat_idx, gray.shape)

print(f"Max value is at Row {row}, Column {col}")
print(img[row])
center = (col,row)
radius = 80
cv.circle(gray,center,radius,(0, 255, 0), thickness=3, lineType=cv.LINE_AA)
text_padding = 40
text_position = (center[0] - 50, center[1] + radius + text_padding)

cv.putText(
    img=gray,
    text="Target",
    org=text_position,
    fontFace=cv.FONT_HERSHEY_SIMPLEX,
    fontScale=1.0,
    color=(255, 255, 255),  # White text
    thickness=2,
    lineType=cv.LINE_AA
)

# Display the result
cv.imshow('Circle with Label', gray)
cv.waitKey(0)
cv.destroyAllWindows()

Max value is at Row 223, Column 358
[[103 132  69]
 [101 132  69]
 [100 131  68]
 ...
 [113 145  80]
 [115 147  82]
 [116 148  83]]


In [24]:
success = cv.imwrite('photos/cat_highest_brightness_point.jpg',gray)

if success:
    print("Image saved successfully!")
else:
    print('Error: Could not save the image.')
# cv.imshow('Blur Comparison',comparison_img_stack)

Image saved successfully!


### C9 — ROI Redactor
Write a function:
```python
def redact(image, regions):
    ...
```
Where `regions` is a list of `(x1, y1, x2, y2)` tuples. The function should:
1. Work on a **copy** of the image (never modify the original)
2. Replace each region with a **solid black box**
3. Draw a thin red border around each redacted box
4. Return the modified copy

Test it by passing at least 3 different regions.

> **Real-world connection**: This is exactly how face/license plate redaction works in production video pipelines.

In [25]:
def redact(image, regions):
    imgCpy = image.copy()
    for (x1,y1,x2,y2) in regions:
        start_point = (x1,y1)
        end_point = (x2,y2)
        cv.rectangle(imgCpy,start_point,end_point,(0,0,0),-1)
        cv.rectangle(imgCpy,start_point,end_point,(0,0,255),2)
    return imgCpy

In [27]:
import cv2 as cv

img = cv.imread('photos/cat_small.jpg')
regions = [
    (10, 10, 50, 50),
    (100, 100, 150, 150),
    (200, 50, 260, 120),
]

redactedImg = redact(img, regions)
# cv.imshow('Redacted Image',redactedImg)
# cv.waitKey(0)
# cv.destroyAllWindows()

success = cv.imwrite('photos/cat_redacted.jpg',redactedImg)

if success:
    print("Image saved successfully!")
else:
    print('Error: Could not save the image.')
# cv.imshow('Blur Comparison',comparison_img_stack)

Image saved successfully!


### C10 — Color Isolator
Write a script that isolates only **green-colored** regions of an image and turns everything else **grayscale**.

The output should be: green objects in color, rest of image in gray.

**Approach to think about**:
1. Convert to HSV (green has a known Hue range in HSV)
2. Create a binary mask for green regions
3. Use the mask to combine the color image and a grayscale version

> **HSV green range hint**: Hue is roughly 35–85 in OpenCV's 0–179 scale. Look up `cv2.inRange` and `cv2.bitwise_and`.

In [50]:
img_bgr = cv.imread('photos/cat_small.jpg')
img_hsv = cv.cvtColor(img_bgr,cv.COLOR_BGR2HSV)

lower_green = (35,57,72)
upper_green = (85,255,255)
# Green Mask Ready
green_mask = cv.inRange(img_hsv,lower_green,upper_green)




# Invert the mask to select everything except green
background_mask = cv.bitwise_not(green_mask)

# Convert image to a three channel gray
# convert to standard single channel
gray_single = cv.cvtColor(img_bgr, cv.COLOR_BGR2GRAY)

# Convert back to bgr to get three identical channels
gray_three_channel = cv.cvtColor(gray_single,cv.COLOR_GRAY2BGR)

# Isolate the green regions from the color image
green_color_part = cv.bitwise_and(img_bgr,img_bgr,mask=green_mask)

# 5. Isolate the background regions from the gray image
gray_background_part = cv.bitwise_and(gray_three_channel, gray_three_channel, mask=background_mask)

# 6. Combine both parts together
final_output = cv.add(green_color_part, gray_background_part)

cv.imshow('Color Splash Effect', final_output)
cv.waitKey(0)
cv.destroyAllWindows()


In [53]:
success = cv.imwrite('photos/cat_color_isolator.jpg',final_output)

if success:
    print("Image saved successfully!")
else:
    print('Error: Could not save the image.')

Image saved successfully!


### C11 — Mini Slideshow
Write a script that:
1. Reads all `.jpg` files from a folder called `images/`
2. Displays them one by one in a window, like a slideshow
3. Shows the filename and image number (e.g., `"Image 2/5: dog.jpg"`) as text overlay on each image
4. Waits 2 seconds between images automatically
5. Lets the user press `q` to quit early

> **Key concepts**: `os.listdir()`, `cv2.waitKey()` return value, drawing text on frames.

In [6]:
import os
import cv2 as cv

folder_name = "photos"
file_names = os.listdir(folder_name)

i = 0
while i < len(file_names):
    imgPath = os.path.join(folder_name,file_names[i])
    if imgPath.endswith(('jpg','png')):
        img = cv.imread(imgPath)
        # "Image 2/5: dog.jpg"
        label = f"Image {i+1}/{len(file_names)}: {file_names[i]}"
        cv.putText(img,label,(10,20),cv.FONT_HERSHEY_SIMPLEX,0.7,(255,0,0),1,cv.LINE_AA)
        cv.imshow('SlideShow',img)
        if cv.waitKey(2000) & 0xFF == ord('q'):
            break  # Exit the loop
    i += 1
cv.destroyAllWindows()

### C12 — The Before/After Diff Tool
Write a complete script that:
1. Takes **two image files** as input (can be hardcoded paths)
2. Resizes both to the same dimensions if they differ
3. Computes the **absolute pixel difference** between them
4. Amplifies the diff so it's visible (multiply by 5, then clip to 255)
5. Saves a **side-by-side comparison** image: `[Image A | Diff | Image B]` as one file

> **Why this matters**: This is the exact core logic of frame-based anomaly detection. You'll reuse this idea in Phase 4 — just swap "two images" for "two consecutive video frames."

In [ ]:

img1 = cv.imread(r'same_res\cat1.jpg')
img2 = cv.imread(r'same_res\cat2.jpg')

# h,w,c
h1,w1,c1 = img1.shape
h2,w2,c2 = img2.shape

if img1.shape != img2.shape:
    h1,w1,c1 = img1.shape
    h2,w2,c2 = img2.shape
    if h2 > h1:
        new_width = w1 
        aspect_ratio = new_width / w2
        new_height = int(h2 * aspect_ratio)
        resizedImg2 = cv.resize(img2,(new_width,new_height),interpolation=cv.INTER_AREA)
        print(resizedImg2.shape)
        cv.imshow('Resized Image',resizedImg2)
        cv.waitKey(0)
        cv.destroyAllWindows()  
else:
    gray1 = cv.cvtColor(img1,cv.COLOR_BGR2GRAY)
    gray2 = cv.cvtColor(img2,cv.COLOR_BGR2GRAY)
    print("Else")
    diff = cv.absdiff(gray1,gray2)

    alpha = 2.5
    beta = 0
    amplified_dff = cv.convertScaleAbs(diff,alpha=alpha,beta=beta)

    cv.imshow('Amplified Difference',amplified_dff)
    cv.waitKey(0)
    cv.destroyAllWindows()


photos\blur_comparison.jpg
(600, 4000, 3)
photos\cat.jpg
(3458, 5026, 3)
photos\cat_channel_setter.jpg
(600, 800, 3)
photos\cat_color_isolator.jpg
(600, 800, 3)
photos\cat_cropped_300x300.jpg
(300, 300, 3)
photos\cat_highest_brightness_point.jpg
(600, 800, 3)
photos\cat_pixel_setter.jpg
(600, 800, 3)
photos\cat_redacted.jpg
(600, 800, 3)
photos\cat_small.jpg
(600, 800, 3)
photos\image.png
(954, 941, 3)


In [36]:
import cv2 as cv
import numpy as np

img1 = cv.imread(r'same_res\cat1.jpg')
img2 = cv.imread(r'same_res\cat2.jpg')

# Step 1: make dimensions match exactly
if img1.shape != img2.shape:
    h1, w1, _ = img1.shape
    img2 = cv.resize(img2, (w1, h1))   # force exact match, not aspect-ratio resize
    print(f"Resized img2 to {img2.shape}")

# Step 2: compute diff (runs regardless of whether resize happened)
diff = cv.absdiff(img1, img2)

# Step 3: amplify
amplified = cv.convertScaleAbs(diff, alpha=2.5, beta=0)

# Step 4: stack and save
comparison = np.hstack([img1, amplified, img2])
cv.imwrite('comparison.jpg', comparison)

cv.imshow('Comparison', comparison)
cv.waitKey(0)
cv.destroyAllWindows()